In [ ]:
import tonic
import tonic.transforms as T

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import UCF101
from torchvision.transforms import v2

from tqdm import tqdm

from models.basicsnn import BasicSNN
from src.utils.config_utils import setup_config_dictionary
from src.utils.save_utils import save_model

batch_size = 64
epochs = 25

config = setup_config_dictionary()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class VideoOnlyUCF101(UCF101):
    """UCF101 dataset wrapper that only returns video and labels"""   
    def __getitem__(self, idx):
        video, audio, label = super().__getitem__(idx)
        return video, label  # Drop audio completely

In [ ]:
transform = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Or check if separate annotation files exist
train_dataset = VideoOnlyUCF101(
    root=f'{config["DATA_SAVE_DIR"]}/UCF-101',
    annotation_path=f'{config["DATA_SAVE_DIR"]}/ucfTrainTestlist',  # separate train file
    frames_per_clip=16,
    step_between_clips=16,
    frame_rate=None,
    train=True,
    transform=transform,
    output_format='TCHW'  # TCHW: Time, Channels, Height, Width
)

# test_dataset = UCF101(
#     root=f'{config["DATA_SAVE_DIR"]}/UCF-101',
#     annotation_path=f'{config["DATA_SAVE_DIR"]}/ucfTrainTestlist',   # separate test file
#     frames_per_clip=16,
#     step_between_clips=16,
#     frame_rate=None,
#     train=False,
#     transform=transform
# )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_video_batch(video_batch, labels_batch, num_samples=4, frames_to_show=8):
    """
    Visualize video samples from a batch
    
    Args:
        video_batch: tensor of shape (batch_size, T, C, H, W)
        labels_batch: tensor of labels
        num_samples: number of videos to show
        frames_to_show: number of frames per video to display
    """
    batch_size, T, C, H, W = video_batch.shape
    
    # UCF101 class names (you might want to load these from a file)
    ucf101_classes = ['ApplyEyeMakeup', 'ApplyLipstick', 'Archery', 'BabyCrawling', 
                      'BalanceBeam', 'BandMarching', 'BaseballPitch', 'Basketball', 
                      'BasketballDunk', 'BenchPress']  # Add all 101 classes as needed
    
    fig, axes = plt.subplots(num_samples, frames_to_show, figsize=(20, 8))
    
    for i in range(min(num_samples, batch_size)):
        video = video_batch[i]  # Shape: (T, C, H, W)
        label = labels_batch[i].item()
        
        # Select frames to display
        frame_indices = np.linspace(0, T-1, frames_to_show, dtype=int)
        
        for j, frame_idx in enumerate(frame_indices):
            frame = video[frame_idx]  # Shape: (C, H, W)
            
            # Denormalize the frame for visualization
            frame_denorm = denormalize_frame(frame)
            
            # Convert to numpy and transpose for matplotlib (H, W, C)
            frame_np = frame_denorm.permute(1, 2, 0).numpy()
            frame_np = np.clip(frame_np, 0, 1)
            
            if num_samples == 1:
                ax = axes[j]
            else:
                ax = axes[i, j]
                
            ax.imshow(frame_np)
            ax.axis('off')
            
            if j == 0:  # Add label to first frame
                class_name = ucf101_classes[label] if label < len(ucf101_classes) else f"Class_{label}"
                ax.set_title(f'{class_name}', fontsize=10)
    
    plt.tight_layout()
    plt.show()

def denormalize_frame(frame):
    """
    Denormalize a frame using ImageNet normalization stats
    """
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    
    frame_denorm = frame * std + mean
    return frame_denorm


In [ ]:
train_loader = torch.utils.data.DataLoader(
        train_dataset, 
        batch_size=256, 
        shuffle=True,
        collate_fn=tonic.collation.PadTensors(), 
        num_workers=0, 
        pin_memory=False, 
)
    
# test_loader = torch.utils.data.DataLoader(
#     test_dataset, 
#     batch_size=batch_size, 
#     shuffle=False, 
#     num_workers=2, 
#     pin_memory=False,
# )

for video, labels in train_loader:
        print(f"Video shape: {video.shape}, Labels shape: {labels.shape}")
        visualize_video_batch(video, labels, num_samples=3, frames_to_show=8)
        break